# Health Index Crosswalk: Local Authority to CCG

The ONS Health Index is published at local authority level, but the MGSR model needs CCG-level scores for People, Places, and Lives. Since every LSOA belongs to exactly one local authority and one CCG, counting the LSOAs shared between each LA-CCG pair gives a population-proportional weight. Each CCG's score is then a weighted average of the scores of the local authorities that overlap with it.

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_PATH = '../data/'

LOOKUP_FILE = os.path.join(BASE_PATH, 'lookup',
    'LSOA11_CCG21_STP21_LAD21_EN_LU_ae1a442ee397483cab1a31f2e7b24029_-5396817129941649351.csv')
HEALTH_INDEX_FILE = os.path.join(BASE_PATH, 'health_index',
    'healthindexscoresatnationalregionalandlocalauthoritylevelsenglandtimeseries.xlsx')
MASTER_PATH = os.path.join(BASE_PATH, 'master_scaled.csv')

## Crosswalk weights

For each LA-CCG pair, the weight is the proportion of that LA's LSOAs that fall within the CCG. Where a local authority sits entirely within one CCG the weight is 1.0, where it straddles multiple CCGs the LSOAs are split proportionally.

In [3]:
lookup = pd.read_csv(LOOKUP_FILE)
master = pd.read_csv(MASTER_PATH, index_col=0)
master_ccg_list = master['ccg'].unique()

# filter to master CCGs only
lookup = lookup[lookup['CCG21CDH'].isin(master_ccg_list)].copy()
print(f'LSOAs in master CCGs: {len(lookup)}')
print(f'CCGs: {lookup["CCG21CDH"].nunique()}')
print(f'LADs: {lookup["LAD21CD"].nunique()}')

crosswalk = lookup.groupby(['LAD21CD', 'CCG21CDH']).size().reset_index(name='lsoa_count')

la_totals = crosswalk.groupby('LAD21CD')['lsoa_count'].transform('sum')
crosswalk['weight'] = crosswalk['lsoa_count'] / la_totals

crosswalk.columns = ['lad_code', 'ccg', 'lsoa_count', 'weight']

print(f'\nLA-CCG pairs: {len(crosswalk)}')
print(crosswalk.head(5).to_string(index=False))

LSOAs in master CCGs: 27542
CCGs: 73
LADs: 261

LA-CCG pairs: 276
 lad_code ccg  lsoa_count  weight
E06000001 16C          58     1.0
E06000002 16C          86     1.0
E06000003 16C          88     1.0
E06000004 16C         120     1.0
E06000005 16C          65     1.0


## Extract Health Index at local authority level

In [4]:
def extract_health_index(filepath, year, sheet_name):
    df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    data = df.iloc[5:, [0, 1, 2, 3, 4]].copy()
    data.columns = ['lad_code', 'lad_name', 'People', 'Lives', 'Places']
    # keep only lower-tier local authority codes
    data = data[data['lad_code'].astype(str).str.match(r'^E0[6-9]', na=False)].copy()
    data['People'] = pd.to_numeric(data['People'], errors='coerce')
    data['Lives']  = pd.to_numeric(data['Lives'],  errors='coerce')
    data['Places'] = pd.to_numeric(data['Places'], errors='coerce')
    data['year'] = year
    return data

hi_2018 = extract_health_index(HEALTH_INDEX_FILE, 2018, 'Table_6_2018_Index')
hi_2019 = extract_health_index(HEALTH_INDEX_FILE, 2019, 'Table_7_2019_Index')

hi_la = pd.concat([hi_2018, hi_2019], ignore_index=True)
print(f'LA-level shape: {hi_la.shape} | LAs: {hi_la["lad_code"].nunique()} | Years: {hi_la["year"].unique()}')
print(hi_la.head())

LA-level shape: (614, 6) | LAs: 307 | Years: [2018 2019]
    lad_code              lad_name  People  Lives  Places  year
0  E06000001            Hartlepool    96.8   90.0    97.9  2018
1  E06000002         Middlesbrough    89.7   90.2    95.0  2018
2  E06000003  Redcar and Cleveland    97.1   97.1   100.5  2018
3  E06000004      Stockton-on-Tees    97.0   96.7    97.6  2018
4  E06000005            Darlington    99.0   99.9    99.2  2018


## Apply crosswalk

In [5]:
hi_ccg = crosswalk.merge(hi_la[['lad_code', 'year', 'People', 'Lives', 'Places']], 
                          on='lad_code', how='left')

# drop pairs where health index is missing
hi_ccg = hi_ccg.dropna(subset=['People', 'Lives', 'Places'])

# renormalise weights after dropping missing
weight_totals = hi_ccg.groupby(['ccg', 'year'])['weight'].transform('sum')
hi_ccg['weight_norm'] = hi_ccg['weight'] / weight_totals

hi_ccg['People_w'] = hi_ccg['People'] * hi_ccg['weight_norm']
hi_ccg['Lives_w']  = hi_ccg['Lives']  * hi_ccg['weight_norm']
hi_ccg['Places_w'] = hi_ccg['Places'] * hi_ccg['weight_norm']

health_index_ccg = hi_ccg.groupby(['ccg', 'year']).agg(
    People=('People_w', 'sum'),
    Lives=('Lives_w',  'sum'),
    Places=('Places_w', 'sum')
).reset_index()

print(f'CCG-level shape: {health_index_ccg.shape}')
print(f'CCGs: {health_index_ccg["ccg"].nunique()}')
print(health_index_ccg.head(10))

CCG-level shape: (146, 5)
CCGs: 73
   ccg    year      People       Lives  Places
0  00Q  2018.0   97.200000   94.400000    99.7
1  00Q  2019.0   96.000000   94.600000    99.5
2  00R  2018.0   85.800000   88.000000    97.6
3  00R  2019.0   84.800000   88.500000    98.5
4  00T  2018.0   95.200000   96.600000   100.1
5  00T  2019.0   93.100000   95.700000   100.0
6  01E  2018.0  105.966667  105.333333   101.4
7  01E  2019.0  105.066667  105.933333   103.1
8  01G  2018.0   90.600000   91.300000    97.5
9  01G  2019.0   91.000000   91.500000    98.0


## Verify against authors' values

Correlation of 0.98+ across all three domains confirms the crosswalk is working. The small remaining differences reflect that the authors' exact method isn't documented as they may have used a slightly different LA-to-CCG mapping vintage.

In [6]:
master_hi = master[['ccg', 'year', 'People', 'Places', 'Lives']].drop_duplicates()
master_hi = master_hi.sort_values(['ccg', 'year']).reset_index(drop=True)
ours = health_index_ccg.sort_values(['ccg', 'year']).reset_index(drop=True)

compare = ours.merge(master_hi, on=['ccg', 'year'], suffixes=('_ours', '_authors'))
print(f'Matched rows: {len(compare)}')
print()

for domain in ['People', 'Lives', 'Places']:
    diff = (compare[f'{domain}_ours'] - compare[f'{domain}_authors']).abs().mean()
    corr = compare[f'{domain}_ours'].corr(compare[f'{domain}_authors'])
    print(f'{domain}: MAD={diff:.2f}, correlation={corr:.4f}')

print()
sample = compare[compare['year'] == 2018].head(5)
print(sample[['ccg', 'People_ours', 'People_authors',
              'Lives_ours', 'Lives_authors',
              'Places_ours', 'Places_authors']].to_string(index=False))

Matched rows: 132

People: MAD=0.46, correlation=0.9786
Lives: MAD=0.47, correlation=0.9805
Places: MAD=0.16, correlation=0.9896

ccg  People_ours  People_authors  Lives_ours  Lives_authors  Places_ours  Places_authors
00Q    97.200000       97.200000   94.400000      94.400000         99.7            99.7
00R    85.800000       85.800000   88.000000      88.000000         97.6            97.6
00T    95.200000       95.200000   96.600000      96.600000        100.1           100.1
01E   105.966667      105.966667  105.333333     105.333333        101.4           101.4
01G    90.600000       90.600000   91.300000      91.300000         97.5            97.5


In [7]:
ours_ccgs   = set(health_index_ccg['ccg'].unique())
master_ccgs = set(master_hi['ccg'].unique())
print('Missing CCGs:', master_ccgs - ours_ccgs)
print('Count:', len(master_ccgs - ours_ccgs))

Missing CCGs: set()
Count: 0


## Save

In [8]:
output_path = os.path.join(BASE_PATH, 'index', 'health_index_ccg.csv')
health_index_ccg.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Shape: {health_index_ccg.shape}')

Saved: ../data/index/health_index_ccg.csv
Shape: (146, 5)
